In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import pandas as pd

train_df = pd.read_csv("/kaggle/input/wmt-2014-english-german/wmt14_translate_de-en_train.csv", lineterminator='\n')

train_df = train_df.sample(frac=0.2, random_state=42)

train_df

,de,en
3535246,Und einen wichtigen Beitrag zum Umweltschutz!,And a major contribution towards environmental...
1744956,Die bisher mäßige Erfolgsquote kann durch eine...,"The success rate, which has been modest so far..."
3530593,"Die Streifen sind blau, neonorange und pink.","The stripes are neon orange, pink and blue."
241383,An der Universität Tübingen hielt man wenig vo...,Working with Tycho's extensive collection of h...
806754,auf Deutsch: Mr. Tompkins' seltsame Reise durc...,"Mr. Tompkins in Paperback, 1965, Cambridge Uni..."
...,...,...
3960349,Ist das nicht dieselbe Art von Scheinheiligkeit?,Is it not the same kind of hypocrisy?
2274707,Deswegen müssen wir eine andere Art von Gesetz...,That is why we are seeking to devise alternati...
3112628,Aber wir brauchen Fristen und es muss dringend...,But we need deadlines and we need action urgen...
1659600,"Herr Präsident, wir bedauern zwar, daß erst ei...","Mr President, much as we regret that it has co..."


In [40]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.nn as nn
import math
import tqdm
import wandb
from torch import nn, optim
import numpy as np




In [2]:
#german text processing
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'([^\w\s])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['de'] = train_df['de'].apply(clean_text)
train_df['en'] = train_df['en'].apply(clean_text)



train_df

,de,en
3535246,und einen wichtigen beitrag zum umweltschutz !,and a major contribution towards environmental...
1744956,die bisher mäßige erfolgsquote kann durch eine...,"the success rate , which has been modest so fa..."
3530593,"die streifen sind blau , neonorange und pink .","the stripes are neon orange , pink and blue ."
241383,an der universität tübingen hielt man wenig vo...,working with tycho ' s extensive collection of...
806754,auf deutsch : mr . tompkins ' seltsame reise d...,"mr . tompkins in paperback , 1965 , cambridge ..."
...,...,...
3960349,ist das nicht dieselbe art von scheinheiligkeit ?,is it not the same kind of hypocrisy ?
2274707,deswegen müssen wir eine andere art von gesetz...,that is why we are seeking to devise alternati...
3112628,aber wir brauchen fristen und es muss dringend...,but we need deadlines and we need action urgen...
1659600,"herr präsident , wir bedauern zwar , daß erst ...","mr president , much as we regret that it has c..."


In [11]:
# more german text processing
src_tokens_list = [s.split() for s in train_df['en'].tolist()]
tgt_tokens_list = [s.split() for s in train_df['de'].tolist()]

from collections import Counter

src_counter = Counter(tok for sent in src_tokens_list for tok in sent)
tgt_counter = Counter(tok for sent in tgt_tokens_list for tok in sent)

src_vocab = {'<pad>': 0, '<unk>': 1, '<bos>': 2, '<eos>': 3}
for i, (tok, _) in enumerate(src_counter.most_common(), start=4):
    src_vocab[tok] = i

tgt_vocab = {'<pad>': 0, '<unk>': 1, '<bos>': 2, '<eos>': 3}
for i, (tok, _) in enumerate(tgt_counter.most_common(), start=4):
    tgt_vocab[tok] = i

inv_src_vocab = {i: w for w, i in src_vocab.items()}
inv_tgt_vocab = {i: w for w, i in tgt_vocab.items()}

max_len = 16 

def encode_and_pad(tokens, vocab, max_len):

    ids = [vocab.get(tok, vocab['<unk>']) for tok in tokens]

    ids = [vocab['<bos>']] + ids + [vocab['<eos>']]

    ids = ids[:max_len]

    if len(ids) < max_len:
        ids += [vocab['<pad>']] * (max_len - len(ids))
    return ids

src_seqs = [encode_and_pad(tok_list, src_vocab, max_len) for tok_list in src_tokens_list]
tgt_seqs = [encode_and_pad(tok_list, tgt_vocab, max_len) for tok_list in tgt_tokens_list]
# print(src_vocab)
# print(tgt_vocab)

In [13]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
src_tensor = torch.tensor(src_seqs, dtype=torch.long)  
tgt_tensor = torch.tensor(tgt_seqs, dtype=torch.long)

In [54]:

# =========================
# NEW: Positional Encoding
# =========================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x

# =========================
# Attention (UPDATED with masks)
# =========================
class AttentionLayer(nn.Module):
    def __init__(self, d_model, d_k, d_v):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_k)
        self.W_k = nn.Linear(d_model, d_k)
        self.W_v = nn.Linear(d_model, d_v)

    def forward(self, q, k, v, mask=None):
        Q = self.W_q(q)
        K = self.W_k(k)
        V = self.W_v(v)
        scores = (Q @ K.transpose(-1, -2)) / math.sqrt(K.size(-1))
        
        # Apply mask if provided (crucial for Decoder)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        attn = torch.softmax(scores, dim=-1)
        return attn @ V

class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, d_model, d_k, d_v):
        super().__init__()
        self.heads = nn.ModuleList([AttentionLayer(d_model, d_k, d_v) for _ in range(n_heads)])
        self.lin = nn.Linear(n_heads * d_v, d_model)

    def forward(self, q, k, v, mask=None):
        z = torch.cat([head(q, k, v, mask) for head in self.heads], dim=-1)
        return self.lin(z)

# =========================
# FeedForward
# =========================
class FeedForward(nn.Module):
    def __init__(self, d_model, d_lin):
        super().__init__()
        self.ffn = nn.Linear(d_model, d_lin)
        self.act = nn.ReLU()
        self.ffn2 = nn.Linear(d_lin, d_model)

    def forward(self, x):
        return self.ffn2(self.act(self.ffn(x)))

# =========================
# EncoderLayer
# =========================
class EncoderLayer(nn.Module):
    def __init__(self, n_heads, d_model, d_k, d_v, d_lin):
        super().__init__()
        self.mha = MultiHeadAttention(n_heads, d_model, d_k, d_v)
        self.ffn = FeedForward(d_model, d_lin)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, src_mask=None):
        x = self.ln1(x + self.mha(x, x, x, src_mask))
        x = self.ln2(x + self.ffn(x))
        return x

# =========================
# DecoderLayer
# =========================
class DecoderLayer(nn.Module):
    def __init__(self, n_heads, d_model, d_k, d_v, d_lin):
        super().__init__()
        self.self_attn = MultiHeadAttention(n_heads, d_model, d_k, d_v)
        self.cross_attn = MultiHeadAttention(n_heads, d_model, d_k, d_v)
        self.ffn = FeedForward(d_model, d_lin)

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ln3 = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, tgt_mask=None, src_mask=None):
        # Self attention needs the causal mask so it can't look at future tokens
        tgt = self.ln1(tgt + self.self_attn(tgt, tgt, tgt, tgt_mask))
        # Cross attention looks at the encoder's memory
        tgt = self.ln2(tgt + self.cross_attn(tgt, memory, memory, src_mask))
        tgt = self.ln3(tgt + self.ffn(tgt))
        return tgt

# =========================
# Encoder & Decoder
# =========================
class Encoder(nn.Module):
    def __init__(self, vocab_size, n_heads, d_model, d_k, d_v, d_lin, n_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([
            EncoderLayer(n_heads, d_model, d_k, d_v, d_lin)
            for _ in range(n_layers)
        ])

    def forward(self, x, src_mask=None):
        x = self.pos_enc(self.embed(x))
        for layer in self.layers:
            x = layer(x, src_mask)
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, n_heads, d_model, d_k, d_v, d_lin, n_layers):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([
            DecoderLayer(n_heads, d_model, d_k, d_v, d_lin)
            for _ in range(n_layers)
        ])
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, memory, tgt_mask=None, src_mask=None):
        x = self.pos_enc(self.embed(tgt))
        for layer in self.layers:
            x = layer(x, memory, tgt_mask, src_mask)
        return self.out(x)

# =========================
# Transformer
# =========================
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, n_heads, d_model, d_k, d_v, d_lin, n_layers):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, n_heads, d_model, d_k, d_v, d_lin, n_layers)
        self.decoder = Decoder(tgt_vocab_size, n_heads, d_model, d_k, d_v, d_lin, n_layers)

    def generate_causal_mask(self, sz):
        # Prevents the decoder from peeking at future tokens
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
        return ~mask # True for allowed connections, False for masked

    def forward(self, src, tgt):
        # Generate causal mask based on target sequence length
        tgt_seq_len = tgt.size(1)
        tgt_mask = self.generate_causal_mask(tgt_seq_len)

        memory = self.encoder(src)
        return self.decoder(tgt, memory, tgt_mask=tgt_mask)

# =========================
# Training & Testing Loops
# =========================
def test(model, test_loader, epoch, criterion):
    test_avg_loss = 0
    num_correct = 0
    total_tokens = 0

    model.eval()
    batch_bar = tqdm.tqdm(total=len(test_loader), dynamic_ncols=True, leave=False, position=0, desc=f"Validate epoch {epoch}")
    
    with torch.no_grad():
        for src, tgt in test_loader:
            src = src.to(device)
            tgt = tgt.to(torch.long).to(device)

            # Seq2Seq setup: input is everything except last token, label is everything except first token
            tgt_input = tgt[:, :-1]
            tgt_labels = tgt[:, 1:]

            output = model(src, tgt_input)
            
            # Reshape for CrossEntropyLoss: (batch * seq_len, vocab_size)
            output_flat = output.contiguous().view(-1, output.shape[-1])
            labels_flat = tgt_labels.contiguous().view(-1)
            
            loss = criterion(output_flat, labels_flat)
            test_avg_loss += loss.item()

            preds = torch.argmax(output_flat, dim=-1)
            
            # Simple token-level accuracy (ignoring pad tokens is recommended if you have them)
            num_correct += torch.sum((preds == labels_flat).float()).item()
            total_tokens += labels_flat.size(0)

            batch_bar.set_postfix(loss="{:.04f}".format(loss.item()))
            batch_bar.update()

    batch_bar.close()
    test_avg_loss /= len(test_loader)
    test_accuracy = num_correct / total_tokens
    return test_avg_loss, test_accuracy
    
def train(model, train_loader, epoch, optimizer, criterion):
    model.train()  # CRITICAL: Enables Dropout and Batchnorm layers
    train_loss = 0
    num_correct = 0
    total_tokens = 0

    batch_bar = tqdm.tqdm(
        total=len(train_loader), 
        dynamic_ncols=True, 
        leave=False, 
        position=0, 
        desc=f"Train Epoch {epoch}"
    )

    for src, tgt in train_loader:
        # 1. Move data to device
        src = src.to(device)
        tgt = tgt.to(torch.long).to(device)

        # 2. Prepare Teacher Forcing indices
        # Input to decoder: <SOS> word1 word2 (exclude last token)
        # Target for loss: word1 word2 <EOS> (exclude first token)
        tgt_input = tgt[:, :-1]
        tgt_labels = tgt[:, 1:]

        # 3. Forward Pass
        optimizer.zero_grad()
        output = model(src, tgt_input)

        # 4. Calculate Loss
        # Reshape to (Batch * Seq, Vocab) for CrossEntropy
        output_flat = output.contiguous().view(-1, output.shape[-1])
        labels_flat = tgt_labels.contiguous().view(-1)
        loss = criterion(output_flat, labels_flat)

        # 5. Backward Pass and Update
        loss.backward()
        
        # Optional: Gradient clipping helps prevent "exploding gradients" in Transformers
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()

        # 6. Metrics
        train_loss += loss.item()
        preds = torch.argmax(output_flat, dim=-1)
        num_correct += torch.sum((preds == labels_flat).float()).item()
        total_tokens += labels_flat.size(0)

        # Update Progress Bar
        batch_bar.set_postfix(
            loss="{:.04f}".format(loss.item()),
            lr="{:.01e}".format(optimizer.param_groups[0]['lr'])
        )
        batch_bar.update()

    batch_bar.close()
    
    avg_loss = train_loss / len(train_loader)
    avg_acc = num_correct / total_tokens
    
    return avg_loss, avg_acc

def run(num_epochs, model, train_loader, test_loader, optimizer, criterion):
    train_loss_hist, train_acc_hist, test_loss_hist, test_acc_hist = [], [], [], []
    
    for epoch in range(num_epochs):
        # --- TRAINING ---
        train_loss, train_acc = train(model, train_loader, epoch, optimizer, criterion)
        
        # --- VALIDATION ---
        test_loss, test_acc = 0.0, 0.0
        if test_loader is not None:
            test_loss, test_acc = test(model, test_loader, epoch, criterion)
        
        # --- LOGGING ---
        print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Test Loss: {test_loss:.4f} | Test Acc: {test_acc*100:.2f}%")
        
        wandb.log({
            "epoch": epoch,
            "train/loss": train_loss,
            "train/acc": train_acc,
            "test/loss": test_loss,
            "test/acc": test_acc,
            "lr": optimizer.param_groups[0]['lr']
        })

        train_loss_hist.append(train_loss)
        test_loss_hist.append(test_loss)

    # --- FINAL SAVING TO WANDB ---
    # We save encoder and decoder separately as requested
    ENCODER_PATH = "transformer_encoder.pth"
    DECODER_PATH = "transformer_decoder.pth"
    encoder_state = model.module.encoder.state_dict() if hasattr(model, 'module') else model.encoder.state_dict()
    decoder_state = model.module.decoder.state_dict() if hasattr(model, 'module') else model.decoder.state_dict()
    
    torch.save(encoder_state, ENCODER_PATH)
    torch.save(decoder_state, DECODER_PATH)
    
    wandb.save(ENCODER_PATH)
    wandb.save(DECODER_PATH)
    print(f"Successfully uploaded weights to WandB.")

    return train_loss_hist, train_acc_hist, test_loss_hist, test_acc_hist

In [57]:
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch the secret
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

# Login
wandb.login(key=wandb_key)

# Initialize the project
run_wandb = wandb.init(
    project="english-german-translator", 
    config={
        "learning_rate": lr,
        "epochs": num_epochs,
        "batch_size": batch_size,
        "d_model": d_model,
        "n_layers": n_layers,
        "n_heads": n_heads,
        "subset_size": subset_size
    }
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: scienceqiu (scienceqiu-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
import gc

torch.cuda.empty_cache()
gc.collect()
# --- 1. Refined Hyperparameters ---
# Vocabulary sizes are determined by your prep work
src_vocab_size = len(src_vocab)
tgt_vocab_size = len(tgt_vocab)

# Model Architecture
d_model = 256          # Increased from 64 for better representation
n_heads = 8            # d_model should be divisible by n_heads (256/8 = 32)
d_k = 32               # Standard: d_model / n_heads
d_v = 32
d_lin = 1024           # Standard: 4 * d_model
n_layers = 4           # Increased from 3 for deeper feature extraction
max_seq_length = 100   # Sufficient for most English-German sentences

# Training Hyperparameters
batch_size = 32        # Adjusted for d_model size and stability
lr = 3e-4              # A more stable starting point for Adam with Transformers
num_epochs = 20

# --- 2. Model Initialization ---
model = Transformer(
    src_vocab_size=src_vocab_size, 
    tgt_vocab_size=tgt_vocab_size, 
    n_heads=n_heads, 
    d_model=d_model, 
    d_k=d_k, 
    d_v=d_v, 
    d_lin=d_lin, 
    n_layers=n_layers
)
print("HI")

model.to(device)

# --- 3. Optimizer and Loss ---
optimizer = optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
loss_func = nn.CrossEntropyLoss(ignore_index=0) # Assuming 0 is your <PAD> token index

# Calculate Parameter Count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

# --- 4. Data Loading ---
from torch.utils.data import TensorDataset, DataLoader, Subset, random_split
import numpy as np

# 1. Create the full dataset
dataset = TensorDataset(src_tensor, tgt_tensor)

# 2. Define your total working subset (e.g., 10,000 samples)
subset_size = 10000
# It's better to use random indices so the model doesn't just learn 
# the first few chapters of a specific text block
indices = torch.randperm(len(dataset))[:subset_size]
working_subset = Subset(dataset, indices)

# 3. Split the subset into Train (90%) and Test (10%)
train_size = int(0.9 * subset_size)
test_size = subset_size - train_size
train_subset, test_subset = random_split(working_subset, [train_size, test_size])

# 4. Train DataLoader
train_dataloader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True
)

# 5. Test DataLoader
test_dataloader = DataLoader(
    test_subset,
    batch_size=batch_size,
    shuffle=False,      # No need to shuffle validation data
    num_workers=1,
    pin_memory=True,
    persistent_workers=True,
    drop_last=False     # Keep all test samples to get accurate accuracy
)

# --- 5. Training Execution ---
train_loss_hist, train_acc_hist, test_loss_hist, test_acc_hist = run(
    num_epochs=num_epochs, 
    model=model, 
    train_loader=train_dataloader,
    test_loader=test_dataloader, # Now passing the test loader
    optimizer=optimizer, 
    criterion=loss_func,
)

HI
Total Parameters: 382,117,622
Trainable Parameters: 382,117,622


Epoch 0 | Train Loss: 8.2356 | Train Acc: 10.31% | Test Loss: 7.3199 | Test Acc: 13.51%


Epoch 1 | Train Loss: 6.4247 | Train Acc: 15.80% | Test Loss: 7.0408 | Test Acc: 16.62%


Epoch 2 | Train Loss: 5.7460 | Train Acc: 19.17% | Test Loss: 6.9504 | Test Acc: 18.64%


Epoch 3 | Train Loss: 5.1905 | Train Acc: 22.58% | Test Loss: 7.0188 | Test Acc: 19.22%


Epoch 4 | Train Loss: 4.6414 | Train Acc: 26.65% | Test Loss: 7.0189 | Test Acc: 19.39%


Epoch 5 | Train Loss: 4.0600 | Train Acc: 32.08% | Test Loss: 7.0988 | Test Acc: 19.01%


Epoch 6 | Train Loss: 3.4595 | Train Acc: 38.92% | Test Loss: 7.3344 | Test Acc: 19.24%


Epoch 7 | Train Loss: 2.8602 | Train Acc: 47.67% | Test Loss: 7.4994 | Test Acc: 18.44%


Train Epoch 8:  54%|█████▎    | 151/281 [00:39<00:33,  3.84it/s, loss=2.1125, lr=3.0e-04]

In [ ]:
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()  

num_epochs = 30
print("\n=== Training with FP16 (Mixed Precision) ===")

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0.0
    batch_count = 0

    for batch_idx, (src_batch, tgt_batch) in enumerate(train_loader):
        current_batch = batch_idx + 1

        src_batch = src_batch.to(device, non_blocking=True)
        tgt_batch = tgt_batch.to(device, non_blocking=True)

        decoder_input = tgt_batch[:, :-1]
        target = tgt_batch[:, 1:]

        optimizer.zero_grad()

        with autocast(): 
            logits = model(src_batch, decoder_input)
            logits = logits.reshape(-1, logits.size(-1))
            loss = criterion(logits, target.reshape(-1))

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        batch_count += 1

        print(f"\rEpoch {epoch:02d} [{current_batch:04d}/{len(train_loader):04d}]  "
              f"Batch Loss: {loss.item():.4f}", end='')

    avg_loss = total_loss / batch_count
    print(f"\nEpoch {epoch:02d} Complete  |  Avg Loss: {avg_loss:.4f}")


In [ ]:
def translate_sentence(model, sentence, src_vocab, tgt_vocab, inv_tgt_vocab, max_len=15):
    model.eval()
    with torch.no_grad():
        # Preprocess
        tokens = sentence.lower().split()
        src_ids = [src_vocab.get(tok, src_vocab['<unk>']) for tok in tokens]
        src_ids = [src_vocab['<bos>']] + src_ids + [src_vocab['<eos>']]
        src_ids = src_ids[:max_len]
        if len(src_ids) < max_len:
            src_ids += [src_vocab['<pad>']] * (max_len - len(src_ids))
        
        src_tensor = torch.tensor([src_ids], dtype=torch.long).to(device)
        
        # Greedy decoding
        tgt_ids = [tgt_vocab['<bos>']]
        for _ in range(max_len-1):
            tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long).to(device)
            
            with torch.cuda.amp.autocast():
                output = model(src_tensor, tgt_tensor)
            
            next_token = output[0, -1, :].argmax().item()
            if next_token == tgt_vocab['<eos>']:
                break
            tgt_ids.append(next_token)
        
        # Decode
        result = [inv_tgt_vocab.get(id, '<unk>') for id in tgt_ids[1:]]  # Skip <bos>
        return ' '.join(result)

# Test
test_sentence = "hello world"
translation = translate_sentence(model, test_sentence, src_vocab, tgt_vocab, inv_tgt_vocab)
print(f"Input: {test_sentence}")
print(f"Translation: {translation}")